# ABI 7500 Fast Dx: connection and offline data

```{device-card} thermo-fisher-abi7500-fast-dx
```

**Incomplete driver.** `ABI7500FastDx` provides USB connection, identity, and temperature/status queries, checked directly from Ubuntu on one G2.10 instrument. Thermal-run control and live fluorescence remain unimplemented and unverified. See the [hardware validation report](hardware-validation.md) for the tested scope.

| Property | Value |
|---|---|
| Transport | USB through the Hamamatsu camera bridge |
| Plate | 96 wells |
| Available live operations | Connect, identify, query status, disconnect |
| Validation | Direct Ubuntu connection/status; captured/native codec checks and mocked USB |


## Connection and physical setup

The Hamamatsu USB connection carries camera messages and a tunnel for thermal-controller packets. Close the Windows 7500 software and ensure no experiment is running before moving the instrument's USB cable to the PLR computer. Keep the instrument powered and use the existing USB Type-B connection. The EZ-Tap RS-232 device is not part of this USB connection.

Install PLR's USB dependencies and give your account access to the instrument's USB device. Do not run the hardware cells without the required local authorization. The packet examples later in this notebook run offline.


## Open the USB connection

This hardware cell configures the communication bridge and checks identification. The driver warns that the integration is incomplete and identifies the limited hardware-tested scope. It does not upload a thermal program, start heating, home filters, or set lamp power.

In [ ]:
from pylabrobot.thermo_fisher import ABI7500FastDx

instrument = ABI7500FastDx()
await instrument.setup()

## Query identity

The camera identification and thermal firmware version refer to different components. Preserve their complete replies when documenting a hardware test.

In [ ]:
identity = await instrument.get_identity()
identity.camera_description, identity.thermal_version

## Query temperature and sequence status

Temperatures are in Celsius and elapsed time is in seconds. Native indices and hold time may retain previous-run values while inactive; elapsed time can continue advancing. An inactive sequence does not prove successful completion or that the cover is cool.

In [ ]:
status = await instrument.get_status()
{
  "block": status.block_temperature,
  "sample": status.sample_temperature,
  "cover": status.cover_temperature,
  "ambient": status.ambient_temperature,
  "phase": status.phase,
  "elapsed": status.elapsed_time,
}

## Close the connection

`stop()` closes USB and ends keepalives. It is not a command to cancel an instrument run. Following a communication failure, call `stop()` before a fresh `setup()`; the driver does not silently resend commands.

In [ ]:
await instrument.stop()

## Offline packet tools

The remaining cells only use saved bytes and fixtures. They never open an instrument connection.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx.protocol import (
  ThermalPacket,
  ThermalPacketDecoder,
  decode_serial_read,
  encode_serial_read,
  encode_serial_write,
)

## Decode a reference packet

This temperature-query packet was generated by the reference binary in an isolated CPU emulator. It is not a captured instrument response. The sequence value identifies a message, not an experiment run.

In [ ]:
reference = bytes.fromhex("5858ffffffff000f393730300006000000000001e6")
packet = ThermalPacket.decode(reference)
assert packet.command == 0x06
assert packet.sequence == 1
assert packet.payload == b""
assert packet.encode() == reference
packet

## Reassemble fragments

A thermal packet can span multiple tunnel replies. A strict decoder retains incomplete bytes and rejects malformed packets without guessing where the next one begins. `finish()` detects a truncated capture.

In [ ]:
decoder = ThermalPacketDecoder()
assert decoder.feed(reference[:9]) == []
reassembled = decoder.feed(reference[9:])
decoder.finish()
assert reassembled == [packet]
reassembled

## Inspect the USB serial wrapper

The recovered serial-write wrapper uses a four-character space-padded length and writes of at most 64 bytes. The serial-read request includes two bytes for the reply count. Inspect these bytes offline; the startup capture confirms the serial wrapper, but PLR hardware control remains unverified.

In [ ]:
write_chunks = encode_serial_write(reference)
read_request = encode_serial_read(len(reference))
assert write_chunks == (b"RS  21" + reference,)
assert read_request == b"RR  23"
{"write_hex": [chunk.hex() for chunk in write_chunks], "read_request": read_request}

## Separate valid bytes from the USB reply tail

Captured RR replies fill the requested length even when fewer serial bytes are available. The little-endian count identifies the valid bytes; the unused tail can be nonzero. This synthetic example wraps the earlier reference packet with four unused bytes. It illustrates parsing only, not a real instrument response. Unmodified captured examples are covered by the protocol tests.

In [ ]:
wire_reply = len(reference).to_bytes(2, "little") + reference + b"\xff" * 4
serial_bytes = decode_serial_read(wire_reply, requested_size=len(reference) + 4)
assert serial_bytes == reference
{"wire_bytes": len(wire_reply), "valid_serial_bytes": len(serial_bytes)}

## Reconstruct a captured camera frame

The repository contains one compressed CD payload as a regression fixture. Its pixels are little-endian unsigned 16-bit values. `CameraFrameAssembler` preserves every byte and reproduces the native library's crop, including its one-pixel offset. This example uses the original USB read boundaries; it does not identify physical wells or apply calibration.

In [ ]:
import base64
import json
import zlib
from importlib.resources import files

from pylabrobot.thermo_fisher.abi7500_fast_dx import CameraFrameAssembler, CameraGeometry

fixture = json.loads(
  files("pylabrobot.thermo_fisher.abi7500_fast_dx")
  .joinpath("test_data/camera_capture_vector.json")
  .read_text()
)
wire_data = zlib.decompress(base64.b64decode(fixture["payload"]))
geometry = CameraGeometry.decode(bytes.fromhex(fixture["ci_reply_hex"]))
assembler = CameraFrameAssembler(geometry)
offset = 0
for length in fixture["usb_read_lengths"]:
  assembler.feed(wire_data[offset : offset + length])
  offset += length
frame = assembler.finish()
assert frame.wire_data == wire_data
{"width": frame.width, "height": frame.height, "first_pixels": frame.pixel_rows()[0][:8]}

## Read saved vendor filter values

EDS members ending in `.tiff` can contain 96 signed 64-bit well values rather than camera TIFFs. This fixture comes from the completed run; its expected values were checked against the vendor's separate live result log. Indices and integer precision are preserved. These are vendor outputs whose optical processing must still be recovered, so they are not labeled uncorrected pixel sums or calibrated measurements.

For a complete local EDS file, import `read_eds_filter_data` from the same package and call `read_eds_filter_data("/path/to/experiment.eds")`. It returns the available arrays in acquisition order; it does not open the instrument.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import SavedFilterData

saved_fixture = json.loads(
  files("pylabrobot.thermo_fisher.abi7500_fast_dx")
  .joinpath("test_data/saved_filter_vector.json")
  .read_text()
)
saved = SavedFilterData.decode(saved_fixture["member"], bytes.fromhex(saved_fixture["data_hex"]))
assert list(saved.well_values) == saved_fixture["expected_values"]
{
  "stage": saved.stage,
  "cycle": saved.cycle,
  "filter": saved.filter,
  "first_values": saved.well_values[:8],
}

## Remaining hardware work

The reference installation, calibration files, and experiments are backed up. Completed and cancelled vendor runs supply USB captures, 62 camera frames, and matched EDS values. Connection, identity and five status queries succeeded directly from Ubuntu with Windows disconnected; see the [hardware validation report](hardware-validation.md). Program upload/start sequencing, cancellation, lost-acknowledgement reconciliation, synchronized acquisition, and calibrated per-well fluorescence remain unimplemented. The offline thermal-program codec alone is insufficient to run an experiment.